# Overview

This example uploads one scanning probe microscopy run to the platform: the run folder becomes a Sample Set with one Sample per measured position, a Measurement Set with one Measurement per Sample and its Setup, the run's records as files, and one hysteresis-loop Property per Sample.
A run folder is what the instrument exports — `summary.json` with the recipe, the session and one record per measured point, and `loops/` with the raw curves — and re-running the notebook adds only what is missing.

## Install the API client

The samples, measurements and files endpoints are not released yet, so the client is installed from its branch until it merges. Restart the kernel after this cell.

In [ ]:
%pip install -q "git+https://github.com/mat3ra/api-client.git@feature/SOF-8051"

## Set Parameters

- **HOST**: platform the run is uploaded to
- **RUN_DIR**: the run folder beside this notebook — what the instrument exports, with `summary.json` and `loops/` inside it
- **ACCOUNT_SLUG**: account the data belongs to, empty for the default account
- **FILES**: which files to upload per measurement

In [ ]:
import urllib.parse

HOST = "https://platform.mat3ra.com"
RUN_DIR = "run"
ACCOUNT_SLUG = ""
FILES = "records"  # "records": the record JSONs, "all": also the loop arrays and plots, "none": no files

url = urllib.parse.urlsplit(HOST)
address = {
    "host": url.hostname,
    "port": url.port or (443 if url.scheme == "https" else 80),
    "secure": url.scheme == "https",
}

## Authenticate and initialize API client

### Authenticate
Authenticate in the browser (OIDC device flow) or via JupyterLite host injection. Credentials are stored in environment variables.

### Initialize API client
Create an authenticated API client and resolve the owner account ID.

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("api")

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate(**address)

# Imports

In [ ]:
from pathlib import Path

from upload_run import account_id, parse, upload

## Parse the run folder

Read the run folder into the documents the platform stores. Nothing is uploaded yet.

In [ ]:
parsed = parse(Path(RUN_DIR))
file_count = sum(len(files) for files in parsed["files"].values())
print(
    f"wafer {parsed['wafer']}: {len(parsed['samples'])} samples (ordered set) · run {parsed['run']}: "
    f"{len(parsed['measurements'])} measurements (ordered set, one per sample) · {len(parsed['records'])} records "
    f"-> {file_count} files · {len(parsed['images'])} image(s) · {len(parsed['properties'])} samples with a combined "
    f"loop · no curves: {len(parsed['skipped'])} samples"
)

## Select the account

`ACCOUNT_SLUG` re-authenticates the client against that account, so the run is read and written there.

In [ ]:
if ACCOUNT_SLUG:
    client = APIClient.authenticate(account_id=account_id(client, ACCOUNT_SLUG), **address)

## Upload the run

Create the Sample Set and its Samples, the Measurement Set and one Measurement per Sample, the files and the loop Properties.

In [ ]:
upload(client, parsed, files=FILES)

## Find the run in the web app

The run is a folder in the account's Measurements tab, named after the run.

In [ ]:
print(f"Open {HOST}, your account's Measurements tab: {parsed['run']}")

## References

- [Mat3ra REST API](https://docs.mat3ra.com/rest-api/overview/)